# Self-Referencing Target Lineage

**Example: Tracking lineage when a query reads from a table it also writes to**


This example demonstrates how clgraph detects and models self-referencing
targets -- queries that INSERT/MERGE into a table while also reading from
that same table (common in SCD Type 2 and CDC patterns).

Key features demonstrated:
1. Single-statement self-reference detection
2. Two-step SCD2 pipeline with cross-query self-read edges
3. Impact analysis through self-read chains
4. Edge annotations (`statement_order`, `edge_role`)

### Imports

In [ ]:
from clgraph import Pipeline

# ============================================================
# Example 1: Single-Statement Self-Reference
# ============================================================

print("=" * 60)
print("Example 1: Single-Statement Self-Reference")
print("=" * 60)

sql_insert_self_ref = """
INSERT INTO dim_customer
SELECT s.id, s.name, s.city,
       COALESCE(t.is_active, 'Y') AS is_active
FROM staging s
LEFT JOIN dim_customer t ON s.id = t.id
WHERE t.id IS NULL
"""

pipeline_ex1 = Pipeline(
    [("insert_new", sql_insert_self_ref)],
    dialect="bigquery",
)

# 1a. ParsedQuery exposes self_referenced_tables
query = list(pipeline_ex1.table_graph.queries.values())[0]
print("\n1a  ParsedQuery.self_referenced_tables")
print(f"    destination  : {query.destination_table}")
print(f"    source_tables: {query.source_tables}")
print(f"    self_ref     : {query.self_referenced_tables}")

# 1b. Self-read nodes in the column graph
self_read_cols = pipeline_ex1.get_self_read_columns("dim_customer")
print(f"\n1b  Self-read nodes for dim_customer ({len(self_read_cols)} found):")
for col in self_read_cols:
    print(f"    -> {col.full_name}  (node_type={col.node_type}, layer={col.layer})")

# 1c. Compare self-read vs physical output nodes
output_cols = [
    c
    for c in pipeline_ex1.columns.values()
    if c.table_name == "dim_customer" and c.layer == "output"
]
print(f"\n1c  Physical output nodes for dim_customer ({len(output_cols)} found):")
for col in output_cols:
    print(f"    -> {col.full_name}  (node_type={col.node_type}, layer={col.layer})")

print("\n    Self-read nodes represent the PRIOR state read via LEFT JOIN.")
print("    Output nodes represent the NEW rows written by the INSERT.")


# ============================================================
# Example 2: Two-Step SCD2 (MERGE + INSERT)
# ============================================================

print("\n" + "=" * 60)
print("Example 2: Two-Step SCD2 Pipeline")
print("=" * 60)

# Step 1: Close old rows
sql_scd2_merge = """
MERGE INTO dim_customer t
USING staging s ON t.id = s.id AND t.is_active = 'Y'
WHEN MATCHED AND (t.name <> s.name OR t.city <> s.city) THEN
  UPDATE SET t.end_time = current_timestamp(), t.is_active = 'N'
"""

# Step 2: Open new version rows
sql_scd2_insert = """
INSERT INTO dim_customer
SELECT s.id, s.name, s.city,
       current_timestamp() AS start_time,
       TIMESTAMP '9999-12-31 00:00:00' AS end_time,
       COALESCE(t.is_active, 'Y') AS is_active
FROM staging s
LEFT JOIN dim_customer t ON s.id = t.id AND t.is_active = 'Y'
WHERE t.id IS NULL OR (t.name <> s.name OR t.city <> s.city)
"""

pipeline_scd2 = Pipeline(
    [
        ("step1_close_rows", sql_scd2_merge),
        ("step2_new_versions", sql_scd2_insert),
    ],
    dialect="bigquery",
)

# 2a. Topological sort order
sorted_ids = pipeline_scd2.table_graph.topological_sort()
print("\n2a  Topological sort order:")
for i, qid in enumerate(sorted_ids):
    q = pipeline_scd2.table_graph.queries[qid]
    print(f"    {i}. {qid}  ({q.operation.value} -> {q.destination_table})")

# 2b. Cross-query edges connecting Step 1 output to Step 2 self-read
cross_query_edges = [e for e in pipeline_scd2.edges if e.edge_role == "cross_query_self_ref"]
print(f"\n2b  Cross-query self-ref edges ({len(cross_query_edges)} found):")
for edge in cross_query_edges:
    print(f"    {edge.from_node.full_name}")
    print(f"      -> {edge.to_node.full_name}")
    print(f"      edge_role={edge.edge_role}, statement_order={edge.statement_order}")

# 2c. Prior-state-read edges within Step 2
prior_state_edges = [e for e in pipeline_scd2.edges if e.edge_role == "prior_state_read"]
print(f"\n2c  Prior-state-read edges ({len(prior_state_edges)} found):")
for edge in prior_state_edges:
    print(f"    {edge.from_node.full_name} -> {edge.to_node.full_name}")


# ============================================================
# Example 3: Impact Analysis Through Self-Read Chain
# ============================================================

print("\n" + "=" * 60)
print("Example 3: Impact Analysis Through Self-Read Chain")
print("=" * 60)

# 3a. Forward trace: staging.city -> dim_customer.city through both steps
forward_hits = pipeline_scd2.trace_column_forward("staging", "city")
print("\n3a  trace_column_forward('staging', 'city'):")
for col in forward_hits:
    print(f"    -> {col.table_name}.{col.column_name}  (query={col.query_id})")

# 3b. Backward trace: dim_customer.is_active shows self-read chain
backward_hits = pipeline_scd2.trace_column_backward("dim_customer", "is_active")
print("\n3b  trace_column_backward('dim_customer', 'is_active'):")
for col in backward_hits:
    print(
        f"    -> {col.table_name}.{col.column_name}  (query={col.query_id}, type={col.node_type})"
    )

# 3c. get_self_read_columns API
sr_cols = pipeline_scd2.get_self_read_columns("dim_customer")
print(f"\n3c  get_self_read_columns('dim_customer') ({len(sr_cols)} nodes):")
for col in sr_cols:
    print(f"    -> {col.full_name}")


# ============================================================
# Example 4: Edge Annotations (statement_order, edge_role)
# ============================================================

print("\n" + "=" * 60)
print("Example 4: Edge Annotations")
print("=" * 60)

# 4a. Filter edges by edge_role
all_roles = {}
for edge in pipeline_scd2.edges:
    role = edge.edge_role or "(none)"
    all_roles.setdefault(role, []).append(edge)

print("\n4a  Edge counts by edge_role:")
for role, edges in sorted(all_roles.items()):
    print(f"    {role}: {len(edges)} edges")

# 4b. Inspect prior_state_read edges
print("\n4b  Edges with edge_role='prior_state_read':")
for edge in pipeline_scd2.edges:
    if edge.edge_role == "prior_state_read":
        print(f"    {edge.from_node.full_name}")
        print(f"      -> {edge.to_node.full_name}")
        print(f"      query_id={edge.query_id}, statement_order={edge.statement_order}")

# 4c. Inspect cross_query_self_ref edges
print("\n4c  Edges with edge_role='cross_query_self_ref':")
for edge in pipeline_scd2.edges:
    if edge.edge_role == "cross_query_self_ref":
        print(f"    {edge.from_node.full_name}")
        print(f"      -> {edge.to_node.full_name}")
        print(f"      statement_order={edge.statement_order} (reflects topological position)")

# 4d. statement_order on all annotated edges
annotated_edges = [e for e in pipeline_scd2.edges if e.statement_order is not None]
print(f"\n4d  All edges with statement_order set ({len(annotated_edges)} total):")
for edge in annotated_edges:
    print(
        f"    order={edge.statement_order}  role={edge.edge_role}  "
        f"{edge.from_node.full_name} -> {edge.to_node.full_name}"
    )

print("\n" + "=" * 60)
print("Self-Referencing Lineage Examples Complete!")
print("=" * 60)

### Visualize SCD2 Pipeline Lineage

Display the simplified column lineage for the two-step SCD2 pipeline,
showing how self-read nodes connect the MERGE and INSERT steps.

In [ ]:
import shutil

from clgraph import visualize_pipeline_lineage

if shutil.which("dot") is None:
    print("\u26a0\ufe0f  Graphviz not installed. Install with: brew install graphviz")
else:
    print("SCD2 Pipeline - Simplified Lineage:")
    display(visualize_pipeline_lineage(pipeline_scd2.column_graph.to_simplified()))